# Set Up

In [1]:
!pip install -U 'jsonargparse[signatures]>=4.27.7' >/dev/null
!pip install gitignore_parser > /dev/null
!pip install lightning > /dev/null

In [2]:
from google.colab import drive
import os
import sys
import yaml
import torch
import importlib
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm import tqdm
from lightning import seed_everything

# 1. Environment Configuration
if not os.path.exists('/content/drive/MyDrive'):
  drive.mount('/content/drive')

project_root = '/content/drive/MyDrive/FundGitHubProject'
eomt_folder = project_root + '/eomt'

os.chdir(project_root)
if project_root not in sys.path:
    sys.path.insert(0, project_root)
if eomt_folder not in sys.path:
    sys.path.insert(0, eomt_folder) # Insert at the beginning to override pre-installed 'datasets'

from eval.iouEval import iouEval

seed_everything(0, verbose=False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active Device: {device}")

Mounted at /content/drive
Active Device: cuda


In [4]:
import os

source = '/content/drive/MyDrive/FundGitHubProject'
destination = '/ProjectFolder'

if not os.path.exists(destination):
    os.symlink(source, destination)
    print("Shortcut created! Refresh the Files menu on the left to see '/ProjectFolder'.")
else:
    print("Shortcut already exists!")

Shortcut already exists!


# Step 5

### Parse yaml config file and initialize the dictionary variable config

In [62]:
import json
import yaml


# The main configuration is stored in the 'config' variable.
# Let's see what the top-level sections are:
config_path = '/ProjectFolder/eomt/configs/dinov2/coco/panoptic/eomt_base_640_2x.yaml'
with open(config_path, "r") as f:
  config = yaml.safe_load(f)
print("Top-level config keys:", config.keys())


Top-level config keys: dict_keys(['trainer', 'model', 'data'])


### Extracting the model and data parameters from config
We extract the model and data parameters so that we can load the backbone, the dataset and the network.

In [73]:
## Load the Cityscapes DataModule and COCO Backbone
import yaml
import importlib

# 1. Load Cityscapes Data Config
cs_config_path = '/ProjectFolder/eomt/configs/dinov2/cityscapes/semantic/eomt_base_640.yaml'
with open(cs_config_path, "r") as f:
    cs_config = yaml.safe_load(f)

data_path = '/ProjectFolder/eomt/data'
data_module_name, class_name = cs_config["data"]["class_path"].rsplit(".", 1)
data_module = getattr(importlib.import_module(data_module_name), class_name)
data_module_kwargs = cs_config["data"].get("init_args", {})

data = data_module(
    path=data_path,
    batch_size=1,
    num_workers=0,
    check_empty_targets=False,
    **data_module_kwargs
)
# We omit calling .setup() to avoid FileNotFoundError if zip is missing

# 2. Load encoder (from COCO config)
encoder_cfg = config["model"]["init_args"]["network"]["init_args"]["encoder"]
encoder_module_name, encoder_class_name = encoder_cfg["class_path"].rsplit(".", 1)
encoder_cls = getattr(importlib.import_module(encoder_module_name), encoder_class_name)

# FIX: Force the encoder to initialize with COCO's image size (640) to match the pos_embed size in the checkpoint
coco_img_size = 640
encoder = encoder_cls(img_size=coco_img_size, **encoder_cfg.get("init_args", {}))

# 3. Load network (from COCO config)
network_cfg = config["model"]["init_args"]["network"]
network_module_name, network_class_name = network_cfg["class_path"].rsplit(".", 1)
network_cls = getattr(importlib.import_module(network_module_name), network_class_name)
network_kwargs = {k: v for k, v in network_cfg["init_args"].items() if k != "encoder"}

# CRITICAL: COCO Panoptic model has 133 classes natively. We must initialize it with 133,
# even though our data is Cityscapes (19 classes).
coco_num_classes = 133
network = network_cls(
    masked_attn_enabled=False,
    num_classes=coco_num_classes,
    encoder=encoder,
    **network_kwargs,
)
print("Cityscapes DataModule and COCO network re-initialized successfully.")


Cityscapes DataModule and COCO network re-initialized successfully.


In [74]:
import importlib
import warnings
import torch
import logging
from lightning import Trainer

# Suppress the PyTorch Lightning save_hyperparameters warning
warnings.filterwarnings("ignore", message=".*is an instance of `nn.Module`.*")
logging.getLogger("lightning.pytorch").setLevel(logging.WARNING)

# 1. Initialize the PyTorch Lightning wrapper class from the COCO config
model_module_name, model_class_name = config["model"]["class_path"].rsplit(".", 1)
model_cls = getattr(importlib.import_module(model_module_name), model_class_name)

# Extract wrapper arguments from config, ignoring 'network' since we pass our instantiated one
model_kwargs = {k: v for k, v in config["model"].get("init_args", {}).items() if k != "network"}

# COCO Panoptic wrapper requires 'stuff_classes' which we can extract from the data config
stuff_classes = config["data"].get("init_args", {}).get("stuff_classes", [])

coco_num_classes = 133
coco_img_size = 640

# Initialize the Lightning Module wrapper FIRST
lightning_model = model_cls(
    network=network,
    img_size=coco_img_size,
    num_classes=coco_num_classes,
    stuff_classes=stuff_classes,
    **model_kwargs
)
print(f"Initialized {model_class_name} wrapper successfully.")

# 2. Load the raw weights directly into memory
bin_path = "/ProjectFolder/eomt/eomt_weights/eomt_coco.bin"
print(f"Loading raw weights from {bin_path}...")
raw_ckpt = torch.load(bin_path, map_location="cpu")
state_dict = raw_ckpt.get("state_dict", raw_ckpt)

# Load the state dict into our newly initialized Lightning model
lightning_model.load_state_dict(state_dict, strict=False)
print("Weights loaded into the model successfully!")

# Disable plotting during validation to avoid wandb logger errors
lightning_model.plot_semantic = lambda *args, **kwargs: None

# Create a Lightning Trainer
trainer = Trainer(
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=1,
    logger=False,
    enable_checkpointing=False
)

# Run validation using the provided dataset and lightning module
print("Starting official PyTorch Lightning validation on Cityscapes data...")
try:
    trainer.validate(model=lightning_model, datamodule=data)
except Exception as e:
    print(f"\n[Validation Error]: {e}")


Initialized MaskClassificationPanoptic wrapper successfully.
Loading raw weights from /ProjectFolder/eomt/eomt_weights/eomt_coco.bin...
Weights loaded into the model successfully!
Starting official PyTorch Lightning validation on Cityscapes data...


Output()

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.



[Validation Error]: 'int' object is not subscriptable


## The mapping

### The Theory of Class Mapping: Bridging COCO and Cityscapes

When working with Semantic Segmentation, a model predicts a specific "class ID" for every single pixel in an image. The challenge arises because different datasets define their classes differently.

**1. The Vocabulary Mismatch**
*   **COCO (Common Objects in Context):** This is a very broad dataset. Depending on the version (e.g., COCO-Stuff vs. standard COCO), it can have anywhere from 80 to over 130 classes. It includes everything from `person`, `car`, and `dog` to `frisbee` and `hot dog`.
*   **Cityscapes:** This dataset is highly specialized for urban street scenes. It evaluates strictly on **19 specific classes** (like `road`, `sidewalk`, `building`, `car`, `pedestrian`, `traffic sign`, etc.).

**2. The Goal of Mapping**
If your model outputs predictions using the COCO "vocabulary" (e.g., it predicts ID `3` because in COCO, `3` is a `car`), but the Cityscapes evaluation script expects Cityscapes IDs (where `car` might be ID `13`), the evaluation metric (`val_iou_all`) will fail completely. The evaluator thinks your model is finding "road" when it's actually finding "car".

Mapping is the process of creating a dictionary or a lookup table to translate the model's output IDs into the evaluator's expected IDs.

**3. The Edge Cases (Where information is lost or preserved)**
Mapping is rarely a perfect 1-to-1 translation. You will encounter three main scenarios:
*   **Direct Mapping (1-to-1):** COCO has a `car` class, and Cityscapes has a `car` class. We simply translate the ID.
*   **Missing/Irrelevant Classes (Many-to-Ignore):** COCO has a `dog` class, but Cityscapes doesn't care about dogs for its primary metrics. These pixels must be mapped to a special **"ignore index"** (often `255`). During evaluation, any pixel marked as `255` is completely skipped, meaning you aren't penalized for predicting it, nor rewarded.
*   **Granularity Differences (Many-to-One):** COCO might have highly specific classes that Cityscapes lumps together, or vice-versa. Careful decisions must be made so that we map the broader concepts into the target dataset's categories without losing valuable contextual predictions.

**4. The Implementation**
Practically, this is usually implemented as a PyTorch tensor operation or a simple array mapping. We take the raw prediction mask (filled with COCO IDs), pass it through our mapping dictionary/array, and output a new mask (filled with Cityscapes IDs and `255` for ignored classes) before handing it to the `iouEval` function.

In [54]:
from torchvision.datasets import Cityscapes

# Get target Cityscapes classes
target_classes = [cls for cls in Cityscapes.classes if not cls.ignore_in_eval]

# Create a lookup for Cityscapes names to Train IDs
cs_class_names = {cls.name: cls.train_id for cls in target_classes}

# Handle slight naming differences between the datasets
cs_class_names['motorbike'] = cs_class_names['motorcycle']
cs_class_names['stop sign'] = cs_class_names['traffic sign']

# Build the mapping dictionary
coco2cs_mapping = {}
for i, coco_name in enumerate(coco80_classes):
    if coco_name in cs_class_names:
        coco2cs_mapping[i] = cs_class_names[coco_name]
    else:
        coco2cs_mapping[i] = 255  # 255 is the standard 'ignore' index

print(f"Mapping Dictionary created with {len(coco2cs_mapping)} entries!")
print("-" * 60)
print("Preview of the first 15 mappings:")
for k in range(15):
    coco_lbl = coco80_classes[k]
    cs_id = coco2cs_mapping[k]
    cs_lbl = "IGNORE (255)" if cs_id == 255 else [cls.name for cls in target_classes if cls.train_id == cs_id][0]
    print(f"COCO ID {k:2d} ({coco_lbl:13s}) --> Cityscapes ID {cs_id:3d} ({cs_lbl})")

Mapping Dictionary created with 80 entries!
------------------------------------------------------------
Preview of the first 15 mappings:
COCO ID  0 (person       ) --> Cityscapes ID  11 (person)
COCO ID  1 (bicycle      ) --> Cityscapes ID  18 (bicycle)
COCO ID  2 (car          ) --> Cityscapes ID  13 (car)
COCO ID  3 (motorbike    ) --> Cityscapes ID  17 (motorcycle)
COCO ID  4 (aeroplane    ) --> Cityscapes ID 255 (IGNORE (255))
COCO ID  5 (bus          ) --> Cityscapes ID  15 (bus)
COCO ID  6 (train        ) --> Cityscapes ID  16 (train)
COCO ID  7 (truck        ) --> Cityscapes ID  14 (truck)
COCO ID  8 (boat         ) --> Cityscapes ID 255 (IGNORE (255))
COCO ID  9 (traffic light) --> Cityscapes ID   6 (traffic light)
COCO ID 10 (fire hydrant ) --> Cityscapes ID 255 (IGNORE (255))
COCO ID 11 (stop sign    ) --> Cityscapes ID   7 (traffic sign)
COCO ID 12 (parking meter) --> Cityscapes ID 255 (IGNORE (255))
COCO ID 13 (bench        ) --> Cityscapes ID 255 (IGNORE (255))
COCO ID 1

### ः‍☁️ Wait, Do We Actually Need This Mapping?

Based on our investigation and comparing this notebook to **Step 4**, here is a crucial realization:

1. **Step 4 (Zero-Shot Evaluation):** We loaded a model natively trained on COCO. Since its raw outputs were COCO class IDs, we **had** to build a complex mapping function (`bridge_to_cs` using `things_map` and `stuff_map`) to translate those predictions into Cityscapes Train IDs before we could evaluate them.
2. **Step 5 (Current Notebook):** We loaded `eomt_base_640.yaml` from the `cityscapes/semantic` configuration folder. When we checked the `num_classes` attribute of our `lightning_model` and `data` module, they were both exactly **19**.

**Conclusion:**
Because this specific model was instantiated and trained directly for Cityscapes, **it natively outputs the 19 Cityscapes Train IDs**.

While building the `coco2cs_mapping` dictionary above is a fantastic educational exercise to understand how cross-dataset evaluation works under the hood, we don't actually need to apply it for this specific configuration! We can pass the model's predictions directly to the evaluator without any intermediate translation.

In [67]:
import torch
import os
from lightning import Trainer

# Re-load the state dict into our initialized Lightning model just to be sure
bin_path = "/ProjectFolder/eomt/eomt_weights/eomt_coco.bin"
print(f"Loading raw weights from {bin_path}...")
raw_ckpt = torch.load(bin_path, map_location="cpu")
state_dict = raw_ckpt.get("state_dict", raw_ckpt)

lightning_model.load_state_dict(state_dict, strict=False)
print("Weights loaded into the model successfully!")

# Re-initialize the trainer just to be clean
trainer = Trainer(
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=1,
    logger=False,
    enable_checkpointing=False
)

# Run validation with the in-memory loaded weights
print("\nStarting official PyTorch Lightning validation...")
trainer.validate(model=lightning_model, datamodule=data)


Loading raw weights from /ProjectFolder/eomt/eomt_weights/eomt_coco.bin...


RuntimeError: Error(s) in loading state_dict for MaskClassificationSemantic:
	size mismatch for network.encoder.backbone.pos_embed: copying a param with shape torch.Size([1, 1600, 768]) from checkpoint, the shape in current model is torch.Size([1, 4096, 768]).
	size mismatch for network.q.weight: copying a param with shape torch.Size([200, 768]) from checkpoint, the shape in current model is torch.Size([100, 768]).
	size mismatch for network.class_head.weight: copying a param with shape torch.Size([134, 768]) from checkpoint, the shape in current model is torch.Size([20, 768]).
	size mismatch for network.class_head.bias: copying a param with shape torch.Size([134]) from checkpoint, the shape in current model is torch.Size([20]).
	size mismatch for criterion.empty_weight: copying a param with shape torch.Size([134]) from checkpoint, the shape in current model is torch.Size([20]).

In [61]:
import os

print("=== Available Model Weights ===")
weights_dir = "/ProjectFolder/eomt/eomt_weights"
if os.path.exists(weights_dir):
    for f in sorted(os.listdir(weights_dir)):
        if f.endswith('.bin') or f.endswith('.ckpt'):
            print(os.path.join(weights_dir, f))
else:
    print(f"Directory not found: {weights_dir}")

print("\n=== Available COCO Configurations ===")
configs_dir = "/ProjectFolder/eomt/configs"
if os.path.exists(configs_dir):
    for root, dirs, files in os.walk(configs_dir):
        for f in files:
            if 'coco' in root.lower() or 'coco' in f.lower():
                print(os.path.join(root, f))
else:
    print(f"Directory not found: {configs_dir}")

=== Available Model Weights ===
/ProjectFolder/eomt/eomt_weights/eomt_cityscapes.bin
/ProjectFolder/eomt/eomt_weights/eomt_cityscapes.ckpt
/ProjectFolder/eomt/eomt_weights/eomt_coco.bin

=== Available COCO Configurations ===
/ProjectFolder/eomt/configs/dinov2/coco/panoptic/eomt_base_640_2x.yaml


In [75]:
import os

training_dir = '/ProjectFolder/eomt/training'
print(f"=== Python files in {training_dir} ===")
if os.path.exists(training_dir):
    for f in sorted(os.listdir(training_dir)):
        if f.endswith('.py'):
            print(f)
else:
    print(f"Directory not found: {training_dir}")

# Let's peek into how the panoptic wrapper handles validation data
panoptic_file = os.path.join(training_dir, 'mask_classification_panoptic.py')
print(f"\n=== Extracting validation_step from {panoptic_file} ===")
if os.path.exists(panoptic_file):
    with open(panoptic_file, 'r') as f:
        lines = f.readlines()

        in_val_step = False
        lines_to_print = []
        for line in lines:
            if 'def validation_step' in line:
                in_val_step = True

            if in_val_step:
                lines_to_print.append(line)
                # Stop reading after the method ends (simple heuristic: next def or enough lines)
                if len(lines_to_print) > 1 and line.startswith('    def '):
                    lines_to_print.pop() # remove the next def
                    break
                if len(lines_to_print) > 25: # Just get the first chunk
                    lines_to_print.append("    # ... [truncated] ...\n")
                    break

        print("".join(lines_to_print))


=== Python files in /ProjectFolder/eomt/training ===
__init__.py
lightning_module.py
mask_classification_instance.py
mask_classification_loss.py
mask_classification_panoptic.py
mask_classification_semantic.py
two_stage_warmup_poly_schedule.py

=== Extracting validation_step from /ProjectFolder/eomt/training/mask_classification_panoptic.py ===



### Custom Inference Loop (Bypassing Lightning)

Since our model outputs COCO Panoptic format and our dataset provides Cityscapes Semantic format, we cannot use `trainer.validate()`. We must do exactly what we did in Step 4: a custom loop that intercepts the raw predictions and translates them using our mapping dictionary.

In [78]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F

# 1. Setup Data and Model for raw PyTorch inference
data.setup(stage="fit") # Ensure data is setup
val_loader = data.val_dataloader()

lightning_model.eval()
lightning_model.to(device)

print("Fetching a batch...")
batch = next(iter(val_loader))

# The dataloader returns a tuple of images and a tuple of targets.
# We need to stack the tuple of images into a single batched tensor (B, C, H, W)
images = torch.stack(list(batch[0])).to(device)

# Convert to float if it is loaded as uint8 (and scale to [0, 1])
if images.dtype == torch.uint8:
    images = images.float() / 255.0

print(f"Original Image shape: {images.shape}")

# Resize to 640x640 as expected by the COCO model
images_resized = F.interpolate(images, size=(640, 640), mode='bilinear', align_corners=False)
print(f"Resized Image shape: {images_resized.shape}")

# 2. Forward Pass (No gradients needed)
with torch.no_grad():
    # The network returns a tuple of (class_logits, mask_logits)
    outputs = lightning_model.network(images_resized)

print("\n--- Inference Successful! ---")
print(f"Outputs type: {type(outputs)}")
if isinstance(outputs, tuple) and len(outputs) >= 2:
    print(f"Class logits shape: {outputs[0][-1].shape}")
    print(f"Mask logits shape: {outputs[1][-1].shape}")

print("\nNext step: We will process these logits into a final 2D mask and apply our coco2cs_mapping!")

Fetching a batch...
Original Image shape: torch.Size([1, 3, 1024, 2048])
Resized Image shape: torch.Size([1, 3, 640, 640])

--- Inference Successful! ---
Outputs type: <class 'tuple'>
Class logits shape: torch.Size([1, 200, 160, 160])
Mask logits shape: torch.Size([1, 200, 134])

Next step: We will process these logits into a final 2D mask and apply our coco2cs_mapping!
